In [1]:
import calliope
import pandas as pd
from pathlib import Path

# Results are one folder up from notebooks/
results_dir = Path("../outputs")

files = {
    "without_transformer": results_dir / "without_transformer.nc",
    "with_transformer": results_dir / "with_transformer.nc",
}

models = {
    scenario: calliope.read_netcdf(path)
    for scenario, path in files.items()
}

In [2]:
def safe_sum(model, var_name, **selectors):
    """Safely sum a Calliope result variable."""
    if var_name not in model.results:
        return 0
    
    data = model.results[var_name]
    
    for dim, value in selectors.items():
        if dim in data.dims:
            data = data.sel({dim: value})
    
    return float(data.sum())


def safe_value(model, var_name, **selectors):
    """Safely get a single value from a Calliope result variable."""
    if var_name not in model.results:
        return 0
    
    data = model.results[var_name]
    
    for dim, value in selectors.items():
        if dim in data.dims:
            data = data.sel({dim: value})
    
    return float(data.sum())

In [4]:
# Use one model as the source of input assumptions
model = models["without_transformer"]

def get_param(model, var_name, tech):
    if var_name not in model.inputs:
        return None
    
    data = model.inputs[var_name]
    
    if "techs" in data.dims:
        data = data.sel(techs=tech)
    
    try:
        return float(data.sum())
    except Exception:
        return None


technologies = ["pv", "battery", "supply_grid_power"]

rows = []

for tech in technologies:
    rows.append({
        "Technology": tech,
        "Capex (Power) [MXN/kW]": get_param(model, "cost_flow_cap", tech),
        "Capex (Storage) [MXN/kWh]": get_param(model, "cost_storage_cap", tech),
        "Grid Price / Opex [MXN/kWh]": get_param(model, "cost_flow_in", tech),
        "Area per Capacity [m²/kW]": get_param(model, "area_use_per_flow_cap", tech),
        "Max Area [m²]": get_param(model, "area_use_max", tech),
        "Max Power Capacity [kW]": get_param(model, "flow_cap_max", tech),
        "Max Storage Capacity [kWh]": get_param(model, "storage_cap_max", tech),
        "Charging Efficiency [-]": get_param(model, "flow_out_eff", tech),
        "Discharging Efficiency [-]": get_param(model, "flow_in_eff", tech),
        "Storage Loss [-]": get_param(model, "storage_loss", tech),
        "Lifetime [years]": get_param(model, "lifetime", tech),
    })

inputs_table = pd.DataFrame(rows)

inputs_table

,Technology,Capex (Power) [MXN/kW],Capex (Storage) [MXN/kWh],Grid Price / Opex [MXN/kWh],Area per Capacity [m²/kW],Max Area [m²],Max Power Capacity [kW],Max Storage Capacity [kWh],Charging Efficiency [-],Discharging Efficiency [-],Storage Loss [-],Lifetime [years]
0,pv,700.0,0.0,0.00,7.0,7627.0,999.0,0.0,0.00,0.00,0.00,25.0
1,battery,160.0,230.0,0.00,0.0,0.0,inf,inf,0.95,0.95,0.01,20.0
2,supply_grid_power,0.0,0.0,0.32,0.0,0.0,165.0,0.0,0.00,0.00,0.00,0.0


In [5]:
rows = []

for scenario, model in models.items():

    pv_capacity_kw = safe_value(
        model, "flow_cap", techs="pv"
    )

    battery_power_capacity_kw = safe_value(
        model, "flow_cap", techs="battery"
    )

    battery_energy_capacity_kwh = safe_value(
        model, "storage_cap", techs="battery"
    )

    pv_area_m2 = safe_value(
        model, "area_use", techs="pv"
    )

    unmet_demand_gwh = safe_sum(
        model, "unmet_demand"
    ) / 1_000_000

    total_investment_cost = safe_sum(
        model, "cost_investment"
    )

    annualised_investment_cost = (
        safe_sum(model, "cost_investment_flow_cap")
        + safe_sum(model, "cost_investment_storage_cap")
        + safe_sum(model, "cost_investment_area_use")
    )

    variable_operating_cost = safe_sum(
        model, "cost_var"
    )

    total_cost_per_year = safe_sum(
        model, "cost"
    )

    total_demand_kwh = safe_sum(
        model, "flow_in", techs="demand_electricity"
    )

    total_levelised_cost = (
        total_cost_per_year / total_demand_kwh
        if total_demand_kwh != 0 else None
    )

    rows.append({
        "Scenario": scenario,
        "PV Capacity [kW]": pv_capacity_kw,
        "Battery Power Capacity [kW]": battery_power_capacity_kw,
        "Battery Energy Capacity [kWh]": battery_energy_capacity_kwh,
        "PV Area [m²]": pv_area_m2,
        "Unmet Demand [GWh/year]": unmet_demand_gwh,
        "Total Investment Cost": total_investment_cost,
        "Total Cost [/year]": total_cost_per_year,
        "Annualised Investment Cost [/year]": annualised_investment_cost,
        "Variable Operating Cost [/year]": variable_operating_cost,
        "Total Levelised Cost [/kWh]": total_levelised_cost,
    })

comparison_table = pd.DataFrame(rows)

comparison_table

,Scenario,PV Capacity [kW],Battery Power Capacity [kW],Battery Energy Capacity [kWh],PV Area [m²],Unmet Demand [GWh/year],Total Investment Cost,Total Cost [/year],Annualised Investment Cost [/year],Variable Operating Cost [/year],Total Levelised Cost [/kWh]
0,without_transformer,357.678082,100.941454,457.125088,2503.746572,0.0,371664.060056,73147.473940,371664.060056,0,0.205339
1,with_transformer,306.888397,97.557739,436.826633,2148.218779,0.0,398416.688899,79882.140934,398416.688899,0,0.224245
